In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

df = pd.read_excel(
    r"C:\Users\Path Bhimani\Downloads\Solar Power Generation Data.xlsx",
    usecols=[0],
    names=['Power']
)

df['Interval'] = (df.index % 96) + 1
df['Day_Number'] = df.index // 96
df['Day_of_Week'] = df['Day_Number'] % 7
df['Day_of_Year'] = df['Day_Number'] + 1

days_in_month = [31,29,31,30,31,30,31,31,30,31,30,31]
month_boundaries = np.cumsum([0] + days_in_month)

def get_month(day):
    for i in range(12):
        if month_boundaries[i] <= day < month_boundaries[i+1]:
            return i + 1

df['Month'] = df['Day_Number'].apply(get_month)

def get_season(month):
    if month in [12, 1, 2]:
        return 0
    elif month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8, 9]:
        return 2
    else:
        return 3

df['Season'] = df['Month'].apply(get_season)

df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

df['DOW_sin'] = np.sin(2 * np.pi * df['Day_of_Week'] / 7)
df['DOW_cos'] = np.cos(2 * np.pi * df['Day_of_Week'] / 7)

df['DOY_sin'] = np.sin(2 * np.pi * df['Day_of_Year'] / 365)
df['DOY_cos'] = np.cos(2 * np.pi * df['Day_of_Year'] / 365)

df['Time_Fraction'] = df['Interval'] / 96
df['Distance_From_Noon'] = abs(df['Interval'] - 48)

df['Sine_Shape'] = np.sin((df['Interval'] - 24) * (np.pi / 48))
df['Cos_Shape'] = np.cos((df['Interval'] - 24) * (np.pi / 48))

df['Sine_Shape'] = df['Sine_Shape'].clip(lower=0)
df['Cos_Shape'] = df['Cos_Shape'].clip(lower=0)

df['Parabolic_Shape'] = 1 - ((df['Interval'] - 48) / 24) ** 2
df['Parabolic_Shape'] = df['Parabolic_Shape'].clip(lower=0)

df['is_day'] = (df['Sine_Shape'] > 0).astype(int)

df['Lag_1'] = df['Power'].shift(1)
df['Lag_2'] = df['Power'].shift(2)
df['Lag_3'] = df['Power'].shift(3)

df['Lag_1hr'] = df['Power'].shift(4)
df['Lag_2hr'] = df['Power'].shift(8)

df['Same_Time_Yesterday'] = df['Power'].shift(96)
df['Same_Time_2_Days'] = df['Power'].shift(192)

df['Rolling_Mean_4'] = df['Power'].rolling(4).mean()
df['Rolling_Std_4'] = df['Power'].rolling(4).std()
df['Rolling_Mean_8'] = df['Power'].rolling(8).mean()

df['Lag1_x_Sine'] = df['Lag_1'] * df['Sine_Shape']
df['Yesterday_x_Sine'] = df['Same_Time_Yesterday'] * df['Sine_Shape']

df['DOY_x_Sine'] = df['DOY_sin'] * df['Sine_Shape']
df['Season_x_Sine'] = df['Season'] * df['Sine_Shape']

df['Gaussian_Shape'] = np.exp(-((df['Interval'] - 48) ** 2) / (2 * (12 ** 2)))
df['Gaussian_Sharp'] = np.exp(-((df['Interval'] - 48) ** 2) / (2 * (8 ** 2)))
df['Gaussian_Wide'] = np.exp(-((df['Interval'] - 48) ** 2) / (2 * (18 ** 2)))

df['Triangle_Shape'] = 1 - abs(df['Interval'] - 48) / 48
df['Triangle_Shape'] = df['Triangle_Shape'].clip(lower=0)

df['Sine_Squared'] = (np.sin((df['Interval'] - 24) * (np.pi / 48))) ** 2
df['Sine_Squared'] = df['Sine_Squared'].clip(lower=0)

df['Log_Shape'] = np.log1p(df['Interval'])
df['Log_Shape'] = df['Log_Shape'] / df['Log_Shape'].max()

df['Exp_Rise'] = np.exp((df['Interval'] - 48) / 20)
df['Exp_Rise'] = df['Exp_Rise'] / df['Exp_Rise'].max()
df['Exp_Rise'] = df['Exp_Rise'].clip(upper=1)

df['Gaussian_x_Season'] = df['Gaussian_Shape'] * df['Season']
df['Gaussian_x_DOY'] = df['Gaussian_Shape'] * df['DOY_sin']
df['Triangle_x_Sine'] = df['Triangle_Shape'] * df['Sine_Shape']

df['Double_Peak'] = (
    np.exp(-((df['Interval'] - 40) ** 2) / 100) +
    np.exp(-((df['Interval'] - 56) ** 2) / 100)
)
df['Double_Peak'] = df['Double_Peak'] / df['Double_Peak'].max()


x = (df['Interval'] - 48) / 48

df['Poly3'] = (1 - x**3).clip(lower=0)
df['Poly4'] = (1 - x**4).clip(lower=0)
df['Poly5'] = (1 - x**5).clip(lower=0)

df['Poly2_shift'] = (1 - ((df['Interval'] - 44) / 48) ** 2).clip(lower=0)
df['Poly4_shift'] = (1 - ((df['Interval'] - 52) / 48) ** 4).clip(lower=0)

df['Poly_peak'] = ((1 - x**2)**2).clip(lower=0)



df = df.dropna()

test_mask = (df.index >= 27840) & (df.index <= 30815)
train_mask = (df.index < 27840) | (df.index > 30815)

train_df = df[train_mask]
test_df = df[test_mask]

y_train = train_df['Power']
y_test = test_df['Power']

all_features = [
    'Sine_Shape', 

    'Same_Time_Yesterday',

    'Lag1_x_Sine', 'Yesterday_x_Sine', 'Double_Peak',
     

    'Poly3', 'Poly4', 'Poly5',
    'Poly2_shift', 'Poly4_shift', 'Poly_peak'

]

def recursive_forecast(model, df, features, test_df):

    preds = []
    steps_per_day = 96
    test_start_loc = df.index.get_loc(test_df.index[0])

    for i in range(0, len(test_df), steps_per_day):

        chunk_size = min(steps_per_day, len(test_df) - i)
        start_idx = test_start_loc + i

        history = df['Power'].iloc[start_idx-200:start_idx].tolist()

        for step in range(chunk_size):

            current_index = start_idx + step
            row = df.iloc[current_index].copy()

            if 'Lag_1' in features:
                row['Lag_1'] = history[-1]
            if 'Lag_2' in features:
                row['Lag_2'] = history[-2]

            if 'Rolling_Mean_4' in features:
                row['Rolling_Mean_4'] = np.mean(history[-4:])

            X_input = pd.DataFrame([row[features]])

            pred = model.predict(X_input)[0]
            pred = max(0, pred)

            preds.append(pred)
            history.append(pred)

    return np.array(preds)

results = []
best_score = -np.inf
best_features = None


best_features = ['Same_Time_Yesterday', 'Lag1_x_Sine', 'Yesterday_x_Sine', 'Double_Peak']


X_train_best = train_df[best_features]

model = LinearRegression()
model.fit(X_train_best, y_train)

y_pred_best = recursive_forecast(model, df, best_features, test_df)
y_pred_best = np.round(y_pred_best, 1)

print(f"\nFinal Model R²: {r2_score(y_test, y_pred_best):.4f}")

plt.figure(figsize=(16,6))

plt.plot(y_test.values, label='Actual')
plt.plot(y_pred_best, '--', label='Forecast')

plt.title('Solar Power Forecast (Index-Based Features)')
plt.xlabel('Time Steps')
plt.ylabel('Power')

plt.legend()
plt.grid(True)
plt.tight_layout()

plt.show()

results_df = pd.DataFrame({
    'Index': test_df.index,
    'Actual': y_test.values,
    'Forecast': y_pred_best
}, index=test_df.index)

results_df.to_excel('solar_forecast_results_LR.xlsx', index_label='Index')